In [2]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env")

import pandas as pd
from src.db import read_sql

In [3]:
q = read_sql("""
    SELECT DISTINCT ON (stage, check_name) stage, check_name, status, failed_rows
    FROM quality.check_results
    WHERE split = 'corrupt'
    ORDER BY stage, check_name, checked_at DESC""")
q.pivot(index="check_name", columns="stage", values=["status", "failed_rows"])

status       failed_rows      
stage             clean   raw       clean   raw
check_name                                     
constant_sensor    WARN  WARN           6     6
cycle_continuity   PASS  FAIL           0    19
duplicate_key      PASS  FAIL           0    15
null               PASS  FAIL           0  2111
range              PASS  FAIL           0    30
schema             PASS  PASS           0     0

In [4]:
reg = read_sql("SELECT task, algorithm, metrics, trained_at FROM ml.model_registry ORDER BY trained_at DESC")
latest = reg.drop_duplicates(["task", "algorithm"]).reset_index(drop=True)
table = pd.concat([latest[["task", "algorithm"]], pd.json_normalize(latest["metrics"])], axis=1)
table

,task,algorithm,val_rmse,test_rmse,test_nasa_score,cv_rmse,detection_rate,early_alarm_rate,lead_time_median
0,rul,lstm,13.41,14.96,511.7,NaN,NaN,NaN,NaN
1,rul,lightgbm,NaN,16.65,473.5,15.77,NaN,NaN,NaN
2,rul,linear,NaN,22.57,1346.9,NaN,NaN,NaN,NaN
3,anomaly,iforest,NaN,NaN,NaN,NaN,1.0,0.067,66.5
4,anomaly,mahalanobis,NaN,NaN,NaN,NaN,0.6,0.000,25.5


In [5]:
rul = table[table["task"] == "rul"].set_index("algorithm")
base = rul.loc["linear", "test_rmse"]
for algo in ["lightgbm", "lstm"]:
    v = rul.loc[algo, "test_rmse"]
    print(f"{algo}: test RMSE {v} (기준선 {base} 대비 {(base - v) / base * 100:.1f}% 개선)")

lightgbm: test RMSE 16.65 (기준선 22.57 대비 26.2% 개선)
lstm: test RMSE 14.96 (기준선 22.57 대비 33.7% 개선)


In [6]:
read_sql("""SELECT COUNT(DISTINCT batch_id) AS batches, COUNT(*) AS rows
            FROM raw.sensor_readings WHERE split = 'stream'""")

,batches,rows
0,0,0
